# Phase 3: Transcribe the clips with every Whisper size

Runs each model on the 3,200 selected clips and scores them. Results are written **straight to Google Drive, one row at a time**, so if Colab disconnects you just re-run the setup cells and the same model cell: it picks up where it stopped.

**Before you start**
1. On your Mac: `python src/make_colab_bundle.py`, then upload `colab_bundle.zip` (or its parts from `colab_upload/`) to a Drive folder called `accent-bias-asr`.
2. In Colab: **Runtime → Change runtime type → T4 GPU**.
3. Run the setup cells (1–3), then one model cell at a time.

Time estimates are rough guesses for a free T4 GPU.

## 1. Check the GPU

In [ ]:
!nvidia-smi -L || echo "NO GPU: Runtime > Change runtime type > T4 GPU"

## 2. Connect Drive and unpack the bundle

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE = '/content/drive/MyDrive/accent-bias-asr'
RAW = f'{DRIVE}/results/raw'      # transcripts land here, on Drive

import os
os.makedirs(RAW, exist_ok=True)
if not os.path.exists('/content/accent-bias-asr/data/audio_subset'):
    # The bundle may be uploaded whole or in parts (colab_bundle.zip.part_aa, _ab, ...)
    if os.path.exists(f'{DRIVE}/colab_bundle.zip'):
        !cp "{DRIVE}/colab_bundle.zip" /content/colab_bundle.zip
    else:
        !cat "{DRIVE}"/colab_bundle.zip.part_* > /content/colab_bundle.zip
    !mkdir -p /content/accent-bias-asr && unzip -q /content/colab_bundle.zip -d /content/accent-bias-asr
%cd /content/accent-bias-asr
!echo "code version: $(cat VERSION)" && ls data/audio_subset | wc -l

## 3. Install Whisper and jiwer

In [ ]:
!pip install -q openai-whisper jiwer

## 4. Whisper tiny (~10 min), then check it by hand

The plan says to check a handful of outputs by hand before running the big models. Read the examples printed below: do the references and predictions line up? Does the normalized text look sensible?

In [ ]:
!python src/transcribe.py --models tiny --out-dir "{RAW}"
!python src/score.py --raw-dir "{RAW}" --out "{DRIVE}/results/results.csv" --show 10

## 5. Whisper base (~10 min) and small (~20 min)

In [ ]:
!python src/transcribe.py --models base small --out-dir "{RAW}"

## 6. Whisper medium (~45–60 min)

In [ ]:
!python src/transcribe.py --models medium --out-dir "{RAW}"

## 7. Whisper large-v3 (~1.5–2 h)

The longest run. If Colab disconnects or you hit the free GPU limit, come back later, run cells 1–3 again, then this cell: finished clips are skipped.

In [ ]:
!python src/transcribe.py --models large-v3 --out-dir "{RAW}"

## 8. Whisper turbo (~20 min)

In [ ]:
!python src/transcribe.py --models turbo --out-dir "{RAW}"

## 9. Stretch goal: wav2vec 2.0 (~10 min)

A non-Whisper model trained only on LibriSpeech audiobooks. Useful contrast: does a model trained on narrower data show bigger accent gaps?

In [ ]:
!python src/transcribe.py --models wav2vec2 --out-dir "{RAW}"

## 10. Score everything

Writes `My Drive/accent-bias-asr/results/results.csv`: one row per clip × model, with reference, prediction, WER and error counts. Download it into the project's `results/` folder for Phase 4.

In [ ]:
!python src/score.py --raw-dir "{RAW}" --out "{DRIVE}/results/results.csv" --show 5